In [41]:
import json
from PIL import Image, ImageDraw
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from array import array
import cv2
import os
from PIL import Image
import sys
from dotenv import load_dotenv
from IPython.display import display, Image as IPImage
import numpy as np

from sklearn.preprocessing import StandardScaler

In [42]:
from dotenv import load_dotenv
def authenticate():
    '''
    Authenticate
    Authenticates your credentials and creates a client.
    '''
    load_dotenv()
    VISION_KEY = "0iBXNB7UR8uDpjmzScO2TI5W0prTHAH86ISmGYXsA37UXGHmm7yhJQQJ99BCACi5YpzXJ3w3AAAFACOGopTW"
    VISION_ENDPOINT = "https://rares.cognitiveservices.azure.com/"
    subscription_key =VISION_KEY
    endpoint =VISION_ENDPOINT
    credentials = CognitiveServicesCredentials(subscription_key)
    computervision_client = ComputerVisionClient(endpoint, credentials)
    '''
    END - Authenticate
    '''
    return computervision_client

In [43]:
computervision_client = authenticate()

In [44]:
def get_classes(image_path):
    with open(image_path, "rb") as image_stream:
        image_analysis = computervision_client.analyze_image_in_stream(image_stream,
                                                                       visual_features=[VisualFeatureTypes.tags,
                                                                                        VisualFeatureTypes.objects])
    print("tags")
    tags = {}
    for tag in image_analysis.tags:
        if tag.name not in tags:
            tags[tag.name] = [tag.confidence]
        else:
            tags[tag.name].append(tag.confidence)
    return tags

In [45]:
import os
from azure.cognitiveservices.vision.computervision.models import ComputerVisionErrorResponseException

def get_predicted_class(image_path, object_searched):
    try:
        with open(image_path, "rb") as image_stream:
            image_analysis = computervision_client.analyze_image_in_stream(
                image_stream,
                visual_features=[VisualFeatureTypes.tags, VisualFeatureTypes.objects]
            )
    except ComputerVisionErrorResponseException as e:
        print(f"Error analyzing image {image_path}: {e.message}")
        return {}, {}

    tags = {}
    for tag in image_analysis.tags:
        if tag.name in object_searched:
            if tag.name not in tags:
                tags[tag.name] = [tag.confidence]
            else:
                tags[tag.name].append(tag.confidence)

    objects = {}
    for ob in image_analysis.objects:
        if ob.object_property in object_searched:
            location = [ob.rectangle.x, ob.rectangle.y, ob.rectangle.w + ob.rectangle.x, ob.rectangle.h + ob.rectangle.y]
            if ob.object_property not in objects:
                objects[ob.object_property] = [(ob.object_property, location, ob.confidence)]
            else:
                objects[ob.object_property].append((ob.object_property, location, ob.confidence))

    return tags, objects

In [46]:
#setting tags for bike items
computervision_client = authenticate()
bycicle_obj = ["bicycle", "bike", "cycle", "bicycle wheel", "bicycle tire", "bicycle frame", "bicycle handlebar",
               "bicycle seat", "bicycle pedal", "bicycle chain", "bicycle saddle", "bicycle fork"]

In [47]:
get_predicted_class("bikes/bike08.jpg",bycicle_obj)

({'bike': [0.9692027568817139],
  'bicycle wheel': [0.9648919105529785],
  'bicycle frame': [0.9402967691421509],
  'bicycle tire': [0.9303791522979736],
  'bicycle handlebar': [0.9090083837509155],
  'bicycle pedal': [0.8845899105072021],
  'bicycle fork': [0.8711687326431274],
  'bicycle': [0.6968545913696289]},
 {})

Acest cod presupune că setul de date de test este organizat într-un director (path_to_test_data) și că numele fișierelor indică clasa adevărată (de exemplu, bike_1.jpg pentru imagini cu biciclete și no_bike_0.jpg pentru imagini fără biciclete). Codul va calcula și afișa raportul de clasificare și matricea de confuzie pentru a evalua performanța algoritmului.

In [20]:
import os

# Define the folder containing the images
folder_path = 'bikes'

# List all files in the folder
image_files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

# Define the bike-related tags
bicycle_obj = ["bicycle", "bike", "cycle", "bicycle wheel", "bicycle tire", "bicycle frame", "bicycle handlebar",
               "bicycle seat", "bicycle pedal", "bicycle chain", "bicycle saddle", "bicycle fork"]

# Loop through each image file and get the predicted class
for image_file in image_files:
    image_path = os.path.join(folder_path, image_file)
    tags, objects = get_predicted_class(image_path, bicycle_obj)

    # Check if any bike-related tags or objects are found
    contains_bike = any(tag in bicycle_obj for tag in tags) or any(obj in bicycle_obj for obj in objects)

    # Print the result
    print(f"Image: {image_file}, Contains bike: {contains_bike}")

Image: traffic04.jpg, Contains bike: False
Image: traffic10.jpg, Contains bike: False
Image: traffic05.jpg, Contains bike: False
Image: traffic07.jpg, Contains bike: False
Image: traffic06.jpg, Contains bike: False
Image: bike08.jpg, Contains bike: True
Image: traffic02.jpg, Contains bike: False
Image: traffic03.jpg, Contains bike: False
Image: bike09.jpg, Contains bike: True
Image: traffic01.jpg, Contains bike: False
Image: bike07.jpg, Contains bike: True
Image: bike06.jpg, Contains bike: True
Image: bike10.jpg, Contains bike: True
Image: bike04.jpg, Contains bike: True
Image: bike05.jpg, Contains bike: True
Image: bike02.jpg, Contains bike: True
Image: traffic08.jpg, Contains bike: False
Image: traffic09.jpg, Contains bike: False
Image: bike03.jpg, Contains bike: True


ComputerVisionErrorResponseException: (429) Requests to the Analyze Image Operation under Computer Vision API (v3.2) have exceeded call rate limit of your current ComputerVision F0 pricing tier. Please retry after 49 seconds. To increase your rate limit switch to a paid tier.

In [24]:
import os
from PIL import Image, ImageDraw

# Define the folder containing the images
folder_path = 'bikes'

# List all files in the folder
image_files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

# Define the bike-related tags
bicycle_obj = ["bicycle", "bike", "cycle", "bicycle wheel", "bicycle tire", "bicycle frame", "bicycle handlebar",
               "bicycle seat", "bicycle pedal", "bicycle chain", "bicycle saddle", "bicycle fork"]

# Loop through each image file and get the predicted class
for image_file in image_files:
    image_path = os.path.join(folder_path, image_file)
    tags, objects = get_predicted_class(image_path, bicycle_obj)

    # Check if any bike-related objects are found
    if any(obj in bicycle_obj for obj in objects):
        # Open the image
        image = Image.open(image_path)
        draw = ImageDraw.Draw(image)

        # Draw bounding boxes around detected bicycles
        for obj in objects:
            if obj in bicycle_obj:
                for _, location, confidence in objects[obj]:
                    draw.rectangle(location, outline="red", width=3)

        # Save or display the image with bounding boxes
        image.show()  # or image.save(f"output/{image_file}")

In [27]:

import json
import os
from PIL import Image, ImageDraw

# JSON data
json_data = [
    {"image_path": "bikes/bike1.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [5, 33, 412, 410]}},
    {"image_path": "bikes/bike02.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [20, 89, 381, 325]}},
    {"image_path": "bikes/bike03.jpg", "type": "with_bike", "nr_of_bikes": 2, "locations": {"1": [65, 147, 194, 396], "2": [158, 144, 348, 408]}},
    {"image_path": "bikes/bike04.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [0, 3, 413, 414]}},
    {"image_path": "bikes/bike05.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [65, 54, 359, 344]}},
    {"image_path": "bikes/bike06.jpg", "type": "with_bike", "nr_of_bikes": 2, "locations": {"1": [60, 140, 206, 394], "2": [152, 146, 361, 406]}},
    {"image_path": "bikes/bike07.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [59, 190, 298, 414]}},
    {"image_path": "bikes/bike08.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [50, 0, 390, 358]}},
    {"image_path": "bikes/bike09.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [50, 0, 390, 358]}},
    {"image_path": "bikes/bike10.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [143, 124, 377, 406]}}
]

# Loop through each image file and draw the ground truth bounding boxes
for item in json_data:
    image_path = item["image_path"]
    locations = item["locations"]

    # Open the image
    image = Image.open(image_path)
    draw = ImageDraw.Draw(image)

    # Draw the bounding boxes
    for bbox in locations.values():
        draw.rectangle(bbox, outline="green", width=3)

    # Save or display the image with bounding boxes
    image.show()  # or image.save(f"output/{os.path.basename(image_path)}")

In [28]:
import json
import os
from PIL import Image, ImageDraw

def calculate_iou(box1, box2):
    """
    Calculate the Intersection over Union (IoU) of two bounding boxes.
    """
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    xi1 = max(x1, x1g)
    yi1 = max(y1, y1g)
    xi2 = min(x2, x2g)
    yi2 = min(y2, y2g)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2g - x1g) * (y2g - y1g)
    union_area = box1_area + box2_area - inter_area

    iou = inter_area / union_area
    return iou

# JSON data with ground truth annotations
ground_truth_data = [
    {"image_path": "bikes/bike1.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [5, 33, 412, 410]}},
    {"image_path": "bikes/bike02.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [20, 89, 381, 325]}},
    {"image_path": "bikes/bike03.jpg", "type": "with_bike", "nr_of_bikes": 2, "locations": {"1": [65, 147, 194, 396], "2": [158, 144, 348, 408]}},
    {"image_path": "bikes/bike04.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [0, 3, 413, 414]}},
    {"image_path": "bikes/bike05.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [65, 54, 359, 344]}},
    {"image_path": "bikes/bike06.jpg", "type": "with_bike", "nr_of_bikes": 2, "locations": {"1": [60, 140, 206, 394], "2": [152, 146, 361, 406]}},
    {"image_path": "bikes/bike07.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [59, 190, 298, 414]}},
    {"image_path": "bikes/bike08.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [50, 0, 390, 358]}},
    {"image_path": "bikes/bike09.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [50, 0, 390, 358]}},
    {"image_path": "bikes/bike10.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [143, 124, 377, 406]}}
]

# Loop through each image file and calculate IoU for each bounding box
for item in ground_truth_data:
    image_path = item["image_path"]
    ground_truth_locations = item["locations"]

    # Get predicted bounding boxes (replace this with your prediction function)
    _, predicted_objects = get_predicted_class(image_path, bicycle_obj)

    # Calculate IoU for each ground truth bounding box
    for gt_id, gt_bbox in ground_truth_locations.items():
        max_iou = 0
        for obj in predicted_objects:
            for _, pred_bbox, _ in predicted_objects[obj]:
                iou = calculate_iou(gt_bbox, pred_bbox)
                max_iou = max(max_iou, iou)
        print(f"Image: {image_path}, Ground Truth Box {gt_id}, Max IoU: {max_iou}")

Image: bikes/bike1.jpg, Ground Truth Box 1, Max IoU: 0.9521738051208221
Image: bikes/bike02.jpg, Ground Truth Box 1, Max IoU: 0.9109725657362344
Image: bikes/bike03.jpg, Ground Truth Box 1, Max IoU: 0.13782722513089005
Image: bikes/bike03.jpg, Ground Truth Box 2, Max IoU: 0.890879094979967
Image: bikes/bike04.jpg, Ground Truth Box 1, Max IoU: 0.9903321881265903
Image: bikes/bike05.jpg, Ground Truth Box 1, Max IoU: 0.8801270558027314
Image: bikes/bike06.jpg, Ground Truth Box 1, Max IoU: 0.2139249536310458
Image: bikes/bike06.jpg, Ground Truth Box 2, Max IoU: 0.8113274336283186
Image: bikes/bike07.jpg, Ground Truth Box 1, Max IoU: 0.8601584939613108
Image: bikes/bike08.jpg, Ground Truth Box 1, Max IoU: 0
Image: bikes/bike09.jpg, Ground Truth Box 1, Max IoU: 0.7184759128646904
Image: bikes/bike10.jpg, Ground Truth Box 1, Max IoU: 0.8853289075795722


In [40]:
import json
import os
from PIL import Image, ImageDraw

def calculate_iou(box1, box2):
    """
    Calculate the Intersection over Union (IoU) of two bounding boxes.
    """
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    xi1 = max(x1, x1g)
    yi1 = max(y1, y1g)
    xi2 = min(x2, x2g)
    yi2 = min(y2, y2g)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2g - x1g) * (y2g - y1g)
    union_area = box1_area + box2_area - inter_area

    iou = inter_area / union_area
    return iou

def calculate_precision_recall(ground_truth_data, predicted_data, iou_threshold=0.5):
    """
    Calculate precision and recall based on IoU values.
    """
    tp = 0  # True Positives
    fp = 0  # False Positives
    fn = 0  # False Negatives

    for gt_item, pred_item in zip(ground_truth_data, predicted_data):
        gt_locations = gt_item["locations"]
        pred_locations = pred_item["locations"]

        for gt_id, gt_bbox in gt_locations.items():
            max_iou = 0
            for pred_bbox in pred_locations.values():
                iou = calculate_iou(gt_bbox, pred_bbox)
                max_iou = max(max_iou, iou)

            if max_iou >= iou_threshold:
                tp += 1
            else:
                fn += 1

        for pred_bbox in pred_locations.values():
            max_iou = 0
            for gt_bbox in gt_locations.values():
                iou = calculate_iou(gt_bbox, pred_bbox)
                max_iou = max(max_iou, iou)

            if max_iou < iou_threshold:
                fp += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return precision, recall

# JSON data with ground truth annotations
ground_truth_data = [
    {"image_path": "bikes/bike1.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [5, 33, 412, 410]}},
    {"image_path": "bikes/bike02.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [20, 89, 381, 325]}},
    {"image_path": "bikes/bike03.jpg", "type": "with_bike", "nr_of_bikes": 2, "locations": {"1": [65, 147, 194, 396], "2": [158, 144, 348, 408]}},
    {"image_path": "bikes/bike04.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [0, 3, 413, 414]}},
    {"image_path": "bikes/bike05.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [65, 54, 359, 344]}},
    {"image_path": "bikes/bike06.jpg", "type": "with_bike", "nr_of_bikes": 2, "locations": {"1": [60, 140, 206, 394], "2": [152, 146, 361, 406]}},
    {"image_path": "bikes/bike07.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [59, 190, 298, 414]}},
    {"image_path": "bikes/bike08.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [50, 0, 390, 358]}},
    {"image_path": "bikes/bike09.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [50, 0, 390, 358]}},
    {"image_path": "bikes/bike10.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [143, 124, 377, 406]}}
]

# Updated predicted data
predicted_data = [
    {"image_path": "bikes/bike1.jpg", "locations": {"1": [3, 16, 412, 410]}},
    {"image_path": "bikes/bike02.jpg", "locations": {"1": [9, 90, 366, 321]}},
    {"image_path": "bikes/bike03.jpg", "locations": {"1": [155, 153, 338, 405]}},
    {"image_path": "bikes/bike04.jpg", "locations": {"1": [0, 2, 414, 412]}},
    {"image_path": "bikes/bike05.jpg", "locations": {"1": [66, 36, 349, 335]}},
    {"image_path": "bikes/bike06.jpg", "locations": {"1": [143, 156, 343, 396]}},
    {"image_path": "bikes/bike07.jpg", "locations": {"1": [51, 206, 308, 416]}},
    {"image_path": "bikes/bike08.jpg", "locations": {}},
    {"image_path": "bikes/bike09.jpg", "locations": {"1": [4, 14, 371, 402]}},
    {"image_path": "bikes/bike10.jpg", "locations": {"1": [138, 135, 391, 404], "2": [0, 0, 799, 503], "3": [27, 490, 808, 1008]}}
]

# Calculate precision and recall
precision, recall = calculate_precision_recall(ground_truth_data, predicted_data)
print(f"Precision: {precision}")
print(f"Recall: {recall}")

Precision: 0.8181818181818182
Recall: 0.75


In [50]:
import json
import os
from PIL import Image, ImageDraw

def calculate_iou(box1, box2):
    """
    Calculate the Intersection over Union (IoU) of two bounding boxes.
    """
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    xi1 = max(x1, x1g)
    yi1 = max(y1, y1g)
    xi2 = min(x2, x2g)
    yi2 = min(y2, y2g)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2g - x1g) * (y2g - y1g)
    union_area = box1_area + box2_area - inter_area

    iou = inter_area / union_area
    return iou

def calculate_precision_recall(ground_truth_data, predicted_data, iou_threshold=0.5):
    """
    Calculate precision and recall based on IoU values.
    """
    tp = 0  # True Positives
    fp = 0  # False Positives
    fn = 0  # False Negatives

    for gt_item, pred_item in zip(ground_truth_data, predicted_data):
        gt_locations = gt_item["locations"]
        pred_locations = pred_item["locations"]

        for gt_id, gt_bbox in gt_locations.items():
            max_iou = 0
            for pred_bbox in pred_locations.values():
                iou = calculate_iou(gt_bbox, pred_bbox)
                max_iou = max(max_iou, iou)

            if max_iou >= iou_threshold:
                tp += 1
            else:
                fn += 1

        for pred_bbox in pred_locations.values():
            max_iou = 0
            for gt_bbox in gt_locations.values():
                iou = calculate_iou(gt_bbox, pred_bbox)
                max_iou = max(max_iou, iou)

            if max_iou < iou_threshold:
                fp += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return precision, recall

def calculate_average_precision(ground_truth_data, predicted_data, iou_threshold=0.5):
    """
    Calculate the average precision over all images.
    """
    precisions = []
    recalls = []

    for gt_item, pred_item in zip(ground_truth_data, predicted_data):
        precision, recall = calculate_precision_recall([gt_item], [pred_item], iou_threshold)
        precisions.append(precision)
        recalls.append(recall)

    average_precision = sum(precisions) / len(precisions) if precisions else 0
    average_recall = sum(recalls) / len(recalls) if recalls else 0

    return average_precision, average_recall

# JSON data with ground truth annotations
ground_truth_data = [
    {"image_path": "bikes/bike1.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [5, 33, 412, 410]}},
    {"image_path": "bikes/bike02.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [20, 89, 381, 325]}},
    {"image_path": "bikes/bike03.jpg", "type": "with_bike", "nr_of_bikes": 2, "locations": {"1": [65, 147, 194, 396], "2": [158, 144, 348, 408]}},
    {"image_path": "bikes/bike04.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [0, 3, 413, 414]}},
    {"image_path": "bikes/bike05.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [65, 54, 359, 344]}},
    {"image_path": "bikes/bike06.jpg", "type": "with_bike", "nr_of_bikes": 2, "locations": {"1": [60, 140, 206, 394], "2": [152, 146, 361, 406]}},
    {"image_path": "bikes/bike07.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [59, 190, 298, 414]}},
    {"image_path": "bikes/bike08.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [50, 0, 390, 358]}},
    {"image_path": "bikes/bike09.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [50, 0, 390, 358]}},
    {"image_path": "bikes/bike10.jpg", "type": "with_bike", "nr_of_bikes": 1, "locations": {"1": [143, 124, 377, 406]}}
]

# Updated predicted data
predicted_data = [
    {"image_path": "bikes/bike1.jpg", "locations": {"1": [3, 16, 412, 410]}},
    {"image_path": "bikes/bike02.jpg", "locations": {"1": [9, 90, 366, 321]}},
    {"image_path": "bikes/bike03.jpg", "locations": {"1": [155, 153, 338, 405]}},
    {"image_path": "bikes/bike04.jpg", "locations": {"1": [0, 2, 414, 412]}},
    {"image_path": "bikes/bike05.jpg", "locations": {"1": [66, 36, 349, 335]}},
    {"image_path": "bikes/bike06.jpg", "locations": {"1": [143, 156, 343, 396]}},
    {"image_path": "bikes/bike07.jpg", "locations": {"1": [51, 206, 308, 416]}},
    {"image_path": "bikes/bike08.jpg", "locations": {}},
    {"image_path": "bikes/bike09.jpg", "locations": {"1": [4, 14, 371, 402]}},
    {"image_path": "bikes/bike10.jpg", "locations": {"1": [138, 135, 391, 404], "2": [0, 0, 799, 503], "3": [27, 490, 808, 1008]}}
]

# Calculate average precision and recall
average_precision, average_recall = calculate_average_precision(ground_truth_data, predicted_data)
print(f"Average Precision: {average_precision}")
print(f"Average Recall: {average_recall}")

Average Precision: 0.8333333333333334
Average Recall: 0.8
